[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day09_feature_engineering/day09_notebook.ipynb)

# Day 9 / 42: Feature Engineering
### 42 Days of AI/ML Challenge | #42DaysOfML

---

## What You Will Learn
- What feature engineering is and why models need it
- 5 core techniques: datetime decomposition, distance features, binning, interaction features, aggregation features
- Real Uber-style dataset to practice on
- Before vs After model accuracy comparison
- Practice exercises with solutions

**Concept:** Feature engineering is the process of creating new input variables from raw data to help your model learn patterns it cannot find on its own.

> **Prerequisites:** Day 7 (Data Cleaning) and Day 8 (EDA) notebooks done. You should be comfortable with pandas and basic sklearn.


## Setup: Install and Import Libraries

In [ ]:
# Install if running in Colab
# !pip install pandas numpy scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

---

## The Dataset: Uber-Style Ride Data

We are working with a simulated ride-hailing dataset. It has raw trip information: timestamps, coordinates, fare, distance, passenger count.

**Goal:** Predict whether a trip will have surge pricing (1) or not (0).

The raw features alone do not give the model enough signal. That is where feature engineering comes in.

In [ ]:
# Create the dataset
np.random.seed(42)
n = 2000

df = pd.DataFrame({
    'pickup_datetime':   pd.date_range('2024-01-01', periods=n, freq='30min'),
    'trip_distance_km':  np.random.exponential(5, n).round(2),
    'passenger_count':   np.random.choice([1, 2, 3, 4], n, p=[0.6, 0.2, 0.1, 0.1]),
    'pickup_lat':        np.random.uniform(12.90, 13.10, n).round(4),
    'pickup_lon':        np.random.uniform(77.50, 77.70, n).round(4),
    'dropoff_lat':       np.random.uniform(12.90, 13.10, n).round(4),
    'dropoff_lon':       np.random.uniform(77.50, 77.70, n).round(4),
    'base_fare':         np.random.uniform(50, 500, n).round(2),
})

# Target: surge = 1 during peak hours OR long trips
df['surge'] = (
    (df['pickup_datetime'].dt.hour.isin([8, 9, 17, 18, 19])) |
    (df['trip_distance_km'] > 12)
).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nSurge distribution:")
print(df['surge'].value_counts())
print(f"\nSurge rate: {df['surge'].mean():.1%}")
df.head()

### Baseline: How Does the Model Perform With Raw Features Only?

In [ ]:
# Raw features - no engineering
base_features = ['trip_distance_km', 'passenger_count', 'base_fare']

X_base = df[base_features]
y = df['surge']

X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train_b, y_train)
acc_base = accuracy_score(y_test, rf_base.predict(X_test_b))

print(f"Baseline accuracy (raw features only): {acc_base:.4f} ({acc_base*100:.2f}%)")
print(f"Features used: {base_features}")

---

## Technique 1: Datetime Decomposition

A raw timestamp like `2024-01-08 08:30:00` means nothing to a model. It cannot compare two timestamps and extract the insight "this is a Monday morning rush hour."

You break it apart into components the model can actually use.

**Uber uses this in their surge pricing model.** They extract hour-of-day and day-of-week from timestamps to identify peak demand windows. Their engineering blog documented that time-based features are among the top predictors for demand spikes.

In [ ]:
# Datetime decomposition
df['hour']         = df['pickup_datetime'].dt.hour
df['day_of_week']  = df['pickup_datetime'].dt.dayofweek   # 0=Monday, 6=Sunday
df['month']        = df['pickup_datetime'].dt.month
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['is_peak_hour'] = df['hour'].isin([8, 9, 17, 18, 19]).astype(int)

print("New datetime features created:")
print(df[['pickup_datetime', 'hour', 'day_of_week', 'is_weekend', 'is_peak_hour']].head(8))

# Visualise: surge rate by hour
surge_by_hour = df.groupby('hour')['surge'].mean()

plt.figure(figsize=(12, 4))
surge_by_hour.plot(kind='bar', color=['#E53935' if x > 0.4 else '#1565C0' for x in surge_by_hour])
plt.title('Surge Rate by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour')
plt.ylabel('Surge Rate')
plt.xticks(rotation=0)
plt.axhline(surge_by_hour.mean(), color='orange', linestyle='--', label='Average')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nPeak hours have {surge_by_hour[surge_by_hour.index.isin([8,9,17,18,19])].mean():.1%} surge rate")
print(f"Off-peak hours have {surge_by_hour[~surge_by_hour.index.isin([8,9,17,18,19])].mean():.1%} surge rate")

---

## Technique 2: Distance Features from Coordinates

Raw latitude and longitude columns tell the model almost nothing individually. The model cannot compute the physical distance between pickup and dropoff from four separate coordinate columns.

You compute the **Haversine distance** yourself and hand that to the model as a single meaningful feature.

The Haversine formula calculates the shortest distance between two points on a sphere (Earth) given their latitude and longitude.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate straight-line distance in km between two GPS coordinates.
    Uses the Haversine formula.
    """
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

df['haversine_dist'] = haversine_distance(
    df['pickup_lat'], df['pickup_lon'],
    df['dropoff_lat'], df['dropoff_lon']
).round(3)

print("Haversine distance feature:")
print(df['haversine_dist'].describe().round(3))

# Compare with raw trip_distance_km
correlation = df['haversine_dist'].corr(df['trip_distance_km'])
print(f"\nCorrelation with trip_distance_km: {correlation:.4f}")
print("(Different because haversine = straight-line, trip_distance = road distance)")

# Scatter to visualise
plt.figure(figsize=(6, 5))
plt.scatter(df['haversine_dist'], df['trip_distance_km'], alpha=0.2, s=10, color='steelblue')
plt.xlabel('Haversine Distance (km)')
plt.ylabel('Actual Trip Distance (km)')
plt.title('Straight-Line vs Road Distance', fontweight='bold')
plt.tight_layout()
plt.show()

---

## Technique 3: Binning (Discretization)

Some continuous variables have a non-linear relationship with your target. A model may learn better if you group the values into categories rather than treating them as a continuous scale.

Two types:
- **Equal-width bins** via `pd.cut()`: you define the boundaries
- **Equal-frequency bins** via `pd.qcut()`: each bin has the same number of records

Zomato and Swiggy use delivery time binning in their ETA models. Instead of predicting exact minutes, they classify orders into Fast / Normal / Delayed buckets first.

In [ ]:
# Equal-width binning: domain knowledge defines the boundaries
df['distance_bucket'] = pd.cut(
    df['trip_distance_km'],
    bins=[0, 3, 7, 15, 100],
    labels=['short', 'medium', 'long', 'very_long']
)

# Equal-frequency binning: data-driven quartiles
df['fare_bucket'] = pd.qcut(
    df['base_fare'],
    q=4,
    labels=['budget', 'standard', 'premium', 'luxury']
)

print("Distance bucket distribution:")
print(df['distance_bucket'].value_counts().sort_index())

print("\nFare bucket distribution:")
print(df['fare_bucket'].value_counts().sort_index())

# Surge rate per bucket
print("\nSurge rate by distance bucket:")
print(df.groupby('distance_bucket', observed=True)['surge'].mean().round(3))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('distance_bucket', observed=True)['surge'].mean().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Surge Rate by Distance Bucket', fontweight='bold')
axes[0].set_ylabel('Surge Rate')
axes[0].tick_params(axis='x', rotation=30)

df.groupby('fare_bucket', observed=True)['surge'].mean().plot(
    kind='bar', ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Surge Rate by Fare Bucket', fontweight='bold')
axes[1].set_ylabel('Surge Rate')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---

## Technique 4: Interaction Features

Sometimes two features together tell the model something neither feature alone can.

- `fare_per_km` = base_fare / trip_distance tells you pricing efficiency per kilometre
- `passengers_x_distance` = passenger_count × trip_distance captures combined load

Linear models especially benefit from interaction features because they cannot multiply two features on their own. Tree-based models learn interactions automatically but explicit interaction features can still speed up learning.

In [ ]:
# Ratio feature
df['fare_per_km'] = (df['base_fare'] / (df['trip_distance_km'] + 0.1)).round(3)

# Multiplicative interaction
df['passengers_x_distance'] = (df['passenger_count'] * df['trip_distance_km']).round(3)

# Difference feature
df['lat_diff'] = (df['dropoff_lat'] - df['pickup_lat']).abs().round(4)
df['lon_diff'] = (df['dropoff_lon'] - df['pickup_lon']).abs().round(4)

print("Interaction features:")
print(df[['fare_per_km', 'passengers_x_distance', 'lat_diff', 'lon_diff']].describe().round(3))

# Correlation of interaction features with target
interaction_features = ['fare_per_km', 'passengers_x_distance', 'lat_diff', 'lon_diff']
print("\nCorrelation with surge target:")
for f in interaction_features:
    corr = df[f].corr(df['surge'])
    print(f"  {f:<30}: {corr:+.4f}")

---

## Technique 5: Aggregation Features (Target / Group Statistics)

Aggregation features give each row information about the group it belongs to.

Example: what is the average fare during the same hour of the day across all trips? This tells the model how expensive this hour typically is, which helps predict surge.

**Warning:** Always compute aggregations on training data only, then map to test data. Computing on the full dataset before splitting is data leakage.

In [ ]:
# Correct way: compute on training data, map to all
# For this demonstration we compute on full df since we haven't split yet
# In production: fit on train, transform train+test

# Average fare by hour
hourly_avg_fare = df.groupby('hour')['base_fare'].mean().rename('avg_hourly_fare')
df = df.merge(hourly_avg_fare, on='hour', how='left')

# Average distance by hour
hourly_avg_dist = df.groupby('hour')['trip_distance_km'].mean().rename('avg_hourly_dist')
df = df.merge(hourly_avg_dist, on='hour', how='left')

# Trip count by hour (demand signal)
hourly_trip_count = df.groupby('hour')['base_fare'].count().rename('trips_in_hour')
df = df.merge(hourly_trip_count, on='hour', how='left')

print("Aggregation features created:")
print(df[['hour', 'base_fare', 'avg_hourly_fare', 'avg_hourly_dist', 'trips_in_hour']].head(10).round(2))

# Visualise average fare by hour
plt.figure(figsize=(12, 4))
df.groupby('hour')['avg_hourly_fare'].first().plot(kind='line', marker='o', color='steelblue')
plt.title('Average Fare by Hour (Aggregation Feature)', fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Fare (INR)')
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Before vs After: Model Accuracy Comparison

Now we have all engineered features. Let's compare the model trained on raw features against the one trained on all engineered features.

In [ ]:
# All engineered features (excluding categoricals which need encoding)
engineered_features = [
    # Raw
    'trip_distance_km', 'passenger_count', 'base_fare',
    # Datetime
    'hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'month',
    # Distance
    'haversine_dist', 'lat_diff', 'lon_diff',
    # Interactions
    'fare_per_km', 'passengers_x_distance',
    # Aggregations
    'avg_hourly_fare', 'avg_hourly_dist', 'trips_in_hour'
]

y = df['surge']

# Split
X_eng = df[engineered_features]
X_base = df[base_features]

X_base_train, X_base_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)
X_eng_train, X_eng_test, _, _ = train_test_split(X_eng, y, test_size=0.2, random_state=42)

# Train both models
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_base_train, y_train)
acc_base = accuracy_score(y_test, rf_base.predict(X_base_test))

rf_eng = RandomForestClassifier(n_estimators=100, random_state=42)
rf_eng.fit(X_eng_train, y_train)
acc_eng = accuracy_score(y_test, rf_eng.predict(X_eng_test))

print(f"Accuracy BEFORE feature engineering: {acc_base:.4f} ({acc_base*100:.2f}%)")
print(f"Accuracy AFTER  feature engineering: {acc_eng:.4f} ({acc_eng*100:.2f}%)")
print(f"Improvement:                        +{(acc_eng - acc_base)*100:.2f} percentage points")

# Bar chart comparison
plt.figure(figsize=(7, 4))
bars = plt.bar(['Before Feature Engineering', 'After Feature Engineering'],
               [acc_base, acc_eng],
               color=['#E53935', '#00C853'], width=0.4, edgecolor='white')
for bar, acc in zip(bars, [acc_base, acc_eng]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', fontweight='bold')
plt.ylim(0.5, 1.10)
plt.ylabel('Accuracy')
plt.title('Impact of Feature Engineering on Model Accuracy', fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Importance: What Did the Model Actually Use?

In [ ]:
# Feature importance from the engineered model
importances = pd.Series(
    rf_eng.feature_importances_,
    index=engineered_features
).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#E53935' if imp > 0.1 else '#1565C0' for imp in importances]
importances.plot(kind='barh', color=colors)
plt.title('Feature Importance After Engineering', fontweight='bold')
plt.xlabel('Importance Score')
plt.axvline(0.05, color='orange', linestyle='--', label='5% threshold')
plt.legend()
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(importances.sort_values(ascending=False).head(5).round(4))

print("\nBottom 5 least important features:")
print(importances.sort_values(ascending=True).head(5).round(4))

print("\nObservation: Engineered features like 'is_peak_hour' and 'hour' ")
print("rank higher than raw features. The model confirmed our domain knowledge.")

---

## Real World Problem: The Data Leakage Trap in Aggregation Features

This is one of the most common production mistakes with feature engineering.

When you compute aggregation features (like average fare per hour), you must compute them **on training data only**, then apply the mapping to the test set. If you compute on the full dataset before splitting, your model has already seen test data statistics during training. It will appear accurate in evaluation but fail on truly new data.

The code below shows the correct production-safe way.

In [ ]:
# Production-safe aggregation feature: fit on train, transform both

# Step 1: Split FIRST
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Step 2: Compute aggregation on TRAIN only
hour_to_avg_fare = df_train.groupby('hour')['base_fare'].mean().to_dict()

# Step 3: Map to BOTH train and test using the train-derived values
df_train['avg_fare_by_hour_safe'] = df_train['hour'].map(hour_to_avg_fare)
df_test['avg_fare_by_hour_safe']  = df_test['hour'].map(hour_to_avg_fare)

# Check for NaN in test (hours not seen in train get NaN - handle with fillna)
missing_in_test = df_test['avg_fare_by_hour_safe'].isna().sum()
print(f"Test rows with unseen hours: {missing_in_test}")

if missing_in_test > 0:
    global_avg = df_train['base_fare'].mean()
    df_test['avg_fare_by_hour_safe'] = df_test['avg_fare_by_hour_safe'].fillna(global_avg)
    print(f"Filled {missing_in_test} missing values with global train mean: {global_avg:.2f}")

print("\nLeakage-free aggregation feature created correctly.")
print("Rule: Compute on train, map to train+test using train statistics.")

---

## Summary: 5 Feature Engineering Techniques

| Technique | What You Create | When to Use |
|-----------|----------------|-------------|
| Datetime Decomposition | hour, day_of_week, is_weekend, is_peak | Any dataset with timestamps |
| Distance Features | Haversine distance, lat/lon diffs | Any dataset with GPS coordinates |
| Binning | Buckets from continuous values | When relationship with target is non-linear |
| Interaction Features | Ratios, products, differences | When combined columns are more meaningful |
| Aggregation Features | Group means, counts, sums | When group-level statistics add signal |

**The production rule:** Always compute aggregation features on training data only, then apply to test. Compute on full data = data leakage.

---

## Practice Exercises

Try these before looking at the solutions.

In [ ]:
# EXERCISE 1
# Create a feature called 'is_night' that is 1 for trips between 10pm and 5am, else 0
# Then check: what is the surge rate during night trips vs day trips?

# Your code here:


In [ ]:
# EXERCISE 2
# Create a feature called 'fare_vs_hour_avg' = base_fare / avg_hourly_fare
# This tells you how expensive THIS trip is relative to the typical trip in that hour
# A value > 1 means the trip is more expensive than average for that hour
# Plot its distribution for surge=1 vs surge=0

# Your code here:


In [ ]:
# EXERCISE 3 (Challenge)
# Create a 'trip_efficiency' feature = haversine_dist / trip_distance_km
# This ratio tells you how direct the route was (1.0 = perfectly direct, <1 = detoured)
# Add it to engineered features and see if it improves model accuracy further

# Your code here:


---

## Solutions

In [ ]:
# SOLUTION 1: is_night feature
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)

print("Night trip distribution:")
print(df['is_night'].value_counts())

print("\nSurge rate during NIGHT trips:", df[df['is_night'] == 1]['surge'].mean().round(4))
print("Surge rate during DAY trips:  ", df[df['is_night'] == 0]['surge'].mean().round(4))

In [ ]:
# SOLUTION 2: fare_vs_hour_avg
df['fare_vs_hour_avg'] = (df['base_fare'] / df['avg_hourly_fare']).round(4)

plt.figure(figsize=(9, 4))
df[df['surge'] == 0]['fare_vs_hour_avg'].hist(
    bins=50, alpha=0.6, color='steelblue', label='No Surge', density=True
)
df[df['surge'] == 1]['fare_vs_hour_avg'].hist(
    bins=50, alpha=0.6, color='#E53935', label='Surge', density=True
)
plt.xlabel('Fare vs Hour Average Ratio')
plt.ylabel('Density')
plt.title('Fare Relative to Hour Average: Surge vs No Surge', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print("Avg fare_vs_hour_avg for surge=0:", df[df['surge']==0]['fare_vs_hour_avg'].mean().round(3))
print("Avg fare_vs_hour_avg for surge=1:", df[df['surge']==1]['fare_vs_hour_avg'].mean().round(3))

In [ ]:
# SOLUTION 3: trip_efficiency feature
df['trip_efficiency'] = (df['haversine_dist'] / (df['trip_distance_km'] + 0.01)).round(4)

print("Trip efficiency distribution:")
print(df['trip_efficiency'].describe().round(3))

# Retrain with this new feature added
extended_features = engineered_features + ['trip_efficiency', 'is_night', 'fare_vs_hour_avg']

X_ext = df[extended_features]
y = df['surge']

X_ext_train, X_ext_test, y_train, y_test = train_test_split(X_ext, y, test_size=0.2, random_state=42)

rf_ext = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ext.fit(X_ext_train, y_train)
acc_ext = accuracy_score(y_test, rf_ext.predict(X_ext_test))

print(f"\nAccuracy with extended features: {acc_ext:.4f} ({acc_ext*100:.2f}%)")
print(f"Previous best:                   {acc_eng:.4f} ({acc_eng*100:.2f}%)")

---

## What's Next

**Day 10: Feature Selection**  
You now know how to create features. Tomorrow you will learn how to remove the ones that hurt your model. Filter methods (correlation, chi-square), wrapper methods (RFE), and embedded methods (Lasso) to keep only what matters.

---

**GitHub repo:** https://github.com/VaishnaviJagtap18/42-days-aiml-challenge  
**Follow along on LinkedIn:** #42DaysOfML